These are some possible functions for create_state_matrix (a method of the mru class)
You can paste them in to model.py.

If the function is described as having a triangular input, set n_state_elements to config.n_state_heads * (self.state_head_order * (self.state_head_order - 1)) // 2.
If the function is described as having a full input, set n_state_elements to state_size.
The description means whether state_elements is intended to fill a strictly lower/upper triangular matrix or a full matrix.

In [ ]:
import torch

In [ ]:
import dataclasses

# feign the MRU class for type hinting, remove the type hint when using these function in the implementation
@dataclasses.dataclass
class MRUInfo:
    state_head_order: int

In [ ]:
# uses LU factorization to ensure the determinant is 1
# requires a full input
def create_state_matrix(self: MRUInfo, state_elements: torch.Tensor) -> torch.Tensor:
     input_matrix = state_elements.unflatten(-1, (self.state_head_order, self.state_head_order))
     
     lower_matrix = input_matrix.tril(diagonal = -1)
     upper_matrix = input_matrix.triu(diagonal = 1)

     diagonal = torch.diagonal(state_matrix, dim1 = -2, dim2 = -1)
     diagonal = torch.exp(diagonal - diagonal.mean(dim = -1, keepdim = True)).sqrt()

     diagonal_matrix = torch.zeros_like(state_matrix)
     diagonal_matrix.diagonal(dim1 = -2, dim2 = -1).copy_(diagonal)

     return (lower_matrix + diagonal_matrix) @ (upper_matrix + diagonal_matrix)

In [ ]:
# uses householder product to construct an orthogonal matrix (very slow)
# requires a triangular input
def create_state_matrix(self: MRUInfo, state_elements: torch.Tensor) -> torch.Tensor:
    input_matrix = torch.zeros(
        state_elements.shape[:-1] + (self.state_head_order, self.state_head_order),
        device = state_elements.device,
        dtype = state_elements.dtype
    )

    # construct a lower triangular matrix with ones on the diagonal
    lower_triangular_indices = torch.tril_indices(self.state_head_order, self.state_head_order, offset = -1)

    input_matrix[..., lower_triangular_indices[0], lower_triangular_indices[1]] = state_elements

    input_matrix = input_matrix + torch.eye(self.state_head_order, device = state_elements.device, dtype = state_elements.dtype)

    tau = 2.0 / torch.sum(input_matrix ** 2, dim = -2)

    original_dtype = input_matrix.dtype
    with torch.amp.autocast(device_type = input_matrix.device.type, enabled = False):
        if original_dtype not in (torch.float32, torch.float64):
            input_matrix = input_matrix.float()
        
        if original_dtype not in (torch.float32, torch.float64):
            tau = tau.float()

        state_matrix = torch.linalg.householder_product(state_matrix, tau = tau)

    state_matrix = state_matrix.to(original_dtype)

    return state_matrix

In [ ]:
# uses the cayley transform to construct an orthogonal matrix
# requires a triangular input
def create_state_matrix(self: MRUInfo, state_elements: torch.Tensor) -> torch.Tensor:
    input_matrix = torch.zeros(
        state_elements.shape[:-1] + (self.state_head_order, self.state_head_order),
        device = state_elements.device,
        dtype = state_elements.dtype
    )

    # construct a skew-symmetric matrix
    lower_triangular_indices = torch.tril_indices(self.state_head_order, self.state_head_order, offset = -1)

    input_matrix[..., lower_triangular_indices[0], lower_triangular_indices[1]] = state_elements
    input_matrix = input_matrix - input_matrix.transpose(-2, -1)

    # take the cayley transform
    identity = torch.eye(self.state_head_order, device = state_elements.device, dtype = state_elements.dtype)

    original_dtype = input_matrix.dtype
    with torch.amp.autocast(device_type = input_matrix.device.type, enabled = False):
        if original_dtype not in (torch.float32, torch.float64):
            input_matrix = input_matrix.float()

        state_matrix = torch.linalg.solve(identity - input_matrix, identity + input_matrix)

    state_matrix = state_matrix.to(original_dtype)

    return state_matrix

In [ ]:
# unflatten the input (inverse vectorization) then add the identity (so the operation is stable at initialization)
# no guarantee of orthogonality or determinant 1
# requires a full input
def create_state_matrix(self: MRUInfo, state_elements: torch.Tensor) -> torch.Tensor:
    state_matrix = state_elements.unflatten(-1, (self.state_head_order, self.state_head_order)) + torch.eye(self.state_head_order, device = state_elements.device, dtype = state_elements.dtype)

    return state_matrix

In [ ]:
# use cayley to construct an orthogonal matrix Q, then multiply it by an upper triangular matrix R
# by ensuring the product of the upper triangular matrix's diagonal is 1 we can ensure that QR determinant 1
# requires a full input
def create_state_matrix(self: MRUInfo, state_elements: torch.Tensor) -> torch.Tensor:
    input_matrix = state_elements.unflatten(-1, (self.state_head_order, self.state_head_order))


    upper_matrix = torch.triu(input_matrix, diagonal = 1)
    
    diagonal = torch.diagonal(input_matrix, dim1 = -2, dim2 = -1)
    diagonal = torch.exp(diagonal - diagonal.mean(dim = -1, keepdim = True))

    upper_matrix[..., torch.arange(self.state_head_order), torch.arange(self.state_head_order)] = diagonal
    
    # construct a skew-symmetric matrix
    skew_symmetric_matrix = input_matrix.tril(diagonal = -1)
    skew_symmetric_matrix = skew_symmetric_matrix - skew_symmetric_matrix.transpose(-2, -1)

    # take the cayley transform
    identity = torch.eye(self.state_head_order, device = state_elements.device, dtype = state_elements.dtype)

    original_dtype = input_matrix.dtype
    with torch.amp.autocast(device_type = input_matrix.device.type, enabled = False):
        if original_dtype not in (torch.float32, torch.float64):
            skew_symmetric_matrix = skew_symmetric_matrix.float()
        
        orthogonal_matrix = torch.linalg.solve(identity - skew_symmetric_matrix, identity + skew_symmetric_matrix)
    
    orthogonal_matrix = orthogonal_matrix.to(original_dtype)

    state_matrix = orthogonal_matrix @ upper_matrix

    return state_matrix

In [ ]:
# uses LU to create matrix with determinant 1, then runs a single step of Newton-Schulz iteration to improve the conditioning. 
# Finally, readjust the scale to approximately set the determinant back to 1
# requires a full input
def create_state_matrix(self: MRUInfo, state_elements: torch.Tensor) -> torch.Tensor:
    # step 1: LU parameterization
    input_matrix = state_elements.unflatten(-1, (self.state_head_order, self.state_head_order))
     
    lower_matrix = input_matrix.tril(diagonal = -1)
    upper_matrix = input_matrix.triu(diagonal = 1)

    diagonal = torch.diagonal(state_matrix, dim1 = -2, dim2 = -1)
    diagonal = torch.exp(diagonal - diagonal.mean(dim = -1, keepdim = True)).sqrt()

    diagonal_matrix = torch.zeros_like(state_matrix)
    diagonal_matrix.diagonal(dim1 = -2, dim2 = -1).copy_(diagonal)

    state_matrix = (lower_matrix + diagonal_matrix) @ (upper_matrix + diagonal_matrix)

    # step 2: Newton-Schulz
    identity = torch.eye(self.state_head_order, device = state_elements.device, dtype = state_elements.dtype)
    
    # squared frobenius norm of state_matrix
    input_mag = (state_matrix ** 2).sum(dim = (-2, -1), keepdim = True)
    # decrease the amount of Netwon-Schulz correction with the magnitude of the input, so that the iteration would converge 
    ns_error = identity - state_matrix.transpose(-2, -1) @ state_matrix / input_mag

    error_mag = (ns_error ** 2).sum(dim=(-2, -1), keepdim=True)

    # mean eigenvalue of ns_error / 2
    mu = (self.state_head_order - 1) / (2.0 * self.state_head_order)
    
    # variance of ns_error / 2
    var = torch.clamp((0.25 * error_mag) / self.state_head_order - mu ** 2, min = 0.0)

    # amount to multiply state_matrix by to approximately restore the determinant to 1
    scale = (1.0 / (1.0 + mu)) * torch.exp(var / (2.0 * (1.0 + mu) ** 2))

    return scale * state_matrix @ (identity + 0.5 * ns_error)
